# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > TABLE OF CONTENTS<br><div>  
* [IMPORTS](#1)
* [INTRODUCTION](#2)
    * [CONFIGURATION](#2.1)
    * [CONFIGURATION PARAMETERS](#2.2)    
    * [DATASET COLUMNS](#2.3)
* [PREPROCESSING](#3)
* [ADVERSARIAL CV](#4)
* [EDA AND VISUALS](#5) 
* [DATA TRANSFORMS](#6)
* [MODEL TRAINING](#7)    
* [ENSEMBLE AND SUBMISSION](#8)  
* [PLANNED WAY FORWARD](#9)     

<a id="1"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > IMPORTS<br> <div> 

In [ ]:
%%time 

# Installing select libraries:-
from gc import collect;
from warnings import filterwarnings;
filterwarnings('ignore');
from IPython.display import clear_output;

!pip install -q --upgrade scipy;
!pip install -q category_encoders;
!pip install -q pygwalker

clear_output();
print();
collect();

In [ ]:
%%time

# General library imports:-
from copy import deepcopy;
import pandas as pd;
import numpy as np;
from scipy.stats import mode, kstest, normaltest, shapiro, anderson, jarque_bera;
from collections import Counter;
from itertools import product;
from colorama import Fore, Style, init;
from warnings import filterwarnings;
filterwarnings('ignore');

from tqdm.notebook import tqdm;
import seaborn as sns;
import matplotlib.pyplot as plt;
%matplotlib inline
import pygwalker as pyg;
from pprint import pprint;

print();
collect();
clear_output();

In [ ]:
%%time 

# Importing model and pipeline specifics:-
from category_encoders import OrdinalEncoder, OneHotEncoder;

# Pipeline specifics:-
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler;
from sklearn.decomposition import PCA;
from sklearn.model_selection import (RepeatedStratifiedKFold as RSKF, 
                                     StratifiedKFold as SKF,
                                     KFold, 
                                     RepeatedKFold as RKF, 
                                     cross_val_score);
from sklearn.inspection import permutation_importance, PartialDependenceDisplay as PDD;
from sklearn.feature_selection import mutual_info_classif, RFE;
from sklearn.pipeline import Pipeline, make_pipeline;
from sklearn.base import BaseEstimator, TransformerMixin;
from sklearn.compose import ColumnTransformer;

# ML Model training:-
from sklearn.calibration import CalibrationDisplay as Clb;
from sklearn.metrics import roc_auc_score;
from sklearn.svm import SVC;
from xgboost import XGBClassifier, XGBRegressor;
from lightgbm import LGBMClassifier, LGBMRegressor, log_evaluation;
from catboost import CatBoostRegressor, CatBoostClassifier;
from sklearn.ensemble import (RandomForestRegressor as RFR,
                              ExtraTreesRegressor as ETR,
                              GradientBoostingRegressor as GBR,
                              HistGradientBoostingRegressor as HGBR,
                              RandomForestClassifier as RFC,
                              ExtraTreesClassifier as ETC,
                              GradientBoostingClassifier as GBC,
                              HistGradientBoostingClassifier as HGBC,
                             );
from sklearn.linear_model import LogisticRegression as LC;

# Ensemble and tuning:-
import optuna;
from optuna import Trial, trial, create_study;
from optuna.samplers import TPESampler, CmaEsSampler;
optuna.logging.set_verbosity = optuna.logging.ERROR;

clear_output();
print();
collect();

In [ ]:
%%time 

# Setting rc parameters in seaborn for plots and graphs- 
# Reference - https://matplotlib.org/stable/tutorials/introductory/customizing.html:-
# To alter this, refer to matplotlib.rcParams.keys()

sns.set({"axes.facecolor"       : "#ffffff",
         "figure.facecolor"     : "#ffffff",
         "axes.edgecolor"       : "#000000",
         "grid.color"           : "#ffffff",
         "font.family"          : ['Cambria'],
         "axes.labelcolor"      : "#000000",
         "xtick.color"          : "#000000",
         "ytick.color"          : "#000000",
         "grid.linewidth"       : 0.75,  
         "grid.linestyle"       : "--",
         "axes.titlecolor"      : '#0099e6',
         'axes.titlesize'       : 8.5,
         'axes.labelweight'     : "bold",
         'legend.fontsize'      : 7.0,
         'legend.title_fontsize': 7.0,
         'font.size'            : 7.5,
         'xtick.labelsize'      : 7.5,
         'ytick.labelsize'      : 7.5,        
        });

# Color printing    
def PrintColor(text:str, color = Fore.BLUE, style = Style.BRIGHT):
    "Prints color outputs using colorama using a text F-string";
    print(style + color + text + Style.RESET_ALL); 

# Making sklearn pipeline outputs as dataframe:-
from sklearn import set_config; 
set_config(transform_output = "pandas");
pd.set_option('display.max_columns', 50);
pd.set_option('display.max_rows', 50);

print();
collect();


<a id="2"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > INTRODUCTION<br><div> 

| Version<br>Number | Version Details | Best CV score| Single/ Ensemble|LB score|
| :-: | --- | :-: | :-: |:-:|
| **V1** |* EDA, plots and secondary features<br>* No scaling<br> * Used original data<br>* Tree based ML models and Optuna ensemble<br>* Introduced pygwalker|0.647882|Ensemble<br> Optuna |0.64466|
| **V2** |* EDA, plots and secondary features<br>* No scaling<br> * Used original data<br>* Selected lesser features for the model<br>* Tree based ML models and Optuna ensemble||Ensemble<br> Optuna ||

<a id="2.1"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > CONFIGURATION<br><div> 

In [ ]:
%%time

# Configuration class:-
class CFG:
    "Configuration class for parameters and CV strategy for tuning and training";
    
    # Data preparation:-   
    version_nb         = 2;
    test_req           = "N";
    gpu_switch         = "OFF"; 
    state              = 42;
    target             = ['EC1', 'EC2'];
    episode            = 18;
    path               = f"/kaggle/input/playground-series-s3e{episode}/";
    orig_path          = f"/kaggle/input/ec-mixed-class/";
    
    dtl_preproc_req    = "Y";
    adv_cv_req         = "Y";
    ftre_plots_req     = "Y";
    ftre_imp_req       = "Y";
    
    # Data transforms and scaling:-    
    conjoin_orig_data  = "Y";
    sec_ftre_req       = "Y";
    scale_req          = "N";
    # NOTE---Keep a value here even if scale_req = N, this is used for linear models:-
    scl_method         = "Z"; 
    enc_method         = 'Label';
    ncomp              = 3;
    
    # Model Training:- 
    baseline_req       = "N";
    pstprcs_oof        = "Y";
    pstprcs_train      = "Y";
    ML                 = "Y";
    use_orig_allfolds  = "N";
    n_splits           = 5 ;
    n_repeats          = 1 ;
    nbrnd_erly_stp     = 200 ;
    mdlcv_mthd         = 'RSKF';
    
    # Ensemble:-    
    ensemble_req       = "Y";
    enscv_mthd         = "RSKF";
    metric_obj         = 'maximize';
    ntrials            = 10 if test_req == "Y" else 250;
    
    # Global variables for plotting:-
    grid_specs = {'visible': True, 'which': 'both', 'linestyle': '--', 
                           'color': 'lightgrey', 'linewidth': 0.75};
    title_specs = {'fontsize': 9, 'fontweight': 'bold', 'color': 'tab:blue'};

print();
PrintColor(f"--> Configuration done!\n");
collect();

In [ ]:
%%time 

# Defining functions to be used throughout the code for common tasks:-

# Scaler to be used for continuous columns:- 
all_scalers = {'Robust': RobustScaler(), 
               'Z': StandardScaler(), 
               'MinMax': MinMaxScaler()
              };
scaler      = all_scalers.get(CFG.scl_method);

# Commonly used CV strategies for later usage:-
all_cv= {'KF'  : KFold(n_splits= CFG.n_splits, shuffle = True, random_state= CFG.state),
         'RKF' : RKF(n_splits= CFG.n_splits, n_repeats = CFG.n_repeats, random_state= CFG.state),
         'RSKF': RSKF(n_splits= CFG.n_splits, n_repeats = CFG.n_repeats, random_state= CFG.state),
         'SKF' : SKF(n_splits= CFG.n_splits, shuffle = True, random_state= CFG.state)
        };

# Defining the competition metric:-
def ScoreMetric(ytrue, ypred)-> float:
    """
    This function calculates the metric for the competition. 
    ytrue- ground truth array
    ypred- predictions
    returns - metric value (float)
    """;
    return roc_auc_score(ytrue, ypred);

def PostProcessPred(preds, post_process = "Y"):
    """
    We limit the prediction values between 0.00001 and 0.99999 herewith. 
    This is specifically useful for regressions that could produce out-of-range predictions
    """;
    
    if post_process == "Y": return np.clip(preds, a_min = 0.00001, a_max = 0.999999);
    else: return preds;
    
collect();
print();


<a id="2.2"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > CONFIGURATION PARAMETERS<br><div> 


| Parameter         | Description                                             | Possible value choices|
| ---               | ---                                                     | :-:                   |
|  version_nb       | Version Number                                          | integer               |
|  gpu_switch       | GPU switch                                              | ON/OFF                |
|  state            | Random state for most purposes                          | integer               |
|  target           | Target column name                                      | yield                 |
|  episode          | Episode Number                                          | integer               |
|  path             | Path for input data files                               |                       |
|  orig_path        | Path for input original data files                      |                       |
|  dtl_preproc_req  | Proprocessing required                                  | Y/N                   |    
|  adv_cv_req       | Adversarial CV required                                 | Y/N                   |
|  ftre_plots_req   | Feature plots required                                  | Y/N                   |
|  ftre_imp_req     | Feature importance required                             | Y/N                   |
|  conjoin_orig_data| Conjoin original data                                   | Y/N                   |
|  sec_ftre_req     | Secondary features required                             | Y/N                   |
|  scale_req        | Scaling required                                        | Y/N                   |
|  scl_method       | Scaling method                                          | Z/ Robust/ MinMax     |
|  baseline_req     | Baseline model required                                 | Y/N                   |
|  pstprcs_oof      | Post-process OOF after model training                   | Y/N                   |
|  pstprcs_train    | Post-process OOF during model training for dev-set      | Y/N                   |
|  ML               | Machine Learning Models                                 | Y/N                   |
|  use_orig_allfolds| Use original data across all folds                      | Y/N                   |
|  n_splits         | Number of CV splits                                     | integer               |
|  n_repeats        | Number of CV repeats                                    | integer               |
|  nbrnd_erly_stp   | Number of early stopping rounds                         | integer               |
|  mdl_cv_mthd      | Model CV method name                                    | RKF/ RSKF/ SKF/ KFold |

<a id="2.3"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > DATASET AND COMPETITION DETAILS<br><div>
    
**Data columns**<br>
**Reference** - <br>
https://www.kaggle.com/competitions/playground-series-s3e18/discussion/419646 <br>
https://www.kaggle.com/code/kimtaehun/multi-label-classification-with-complete-eda >br>


- **BertzCT**: Bertz counter (Topological Charge Transfer) value associated with the structural complexity of the molecule.
- **Chi1, Chi1n, Chi1v**: Kier's First-order Molecular Connectivity Index, representing the molecular surface area.
- **Chi2n, Chi2v, Chi3v**: Kier's Second-order and Third-order Connectivity Index, representing the 2nd and 3rd degree connectivity of the molecule.
- **Chi4n**: Kier's Fourth-order Connectivity Index, representing the 4th degree connectivity of the molecule.
- **EState_VSA1, EState_VSA2**: EState-VSA (E-State Value Sum) 1 and 2, representing the electrotopological state contributions of the molecule.
- **ExactMolWt**: Exact molecular weight of the molecule.
- **FpDensityMorgan1, FpDensityMorgan2, FpDensityMorgan3**: Density values of the Morgan fingerprint.
- **HallKierAlpha**: Hall-Kier Alpha value of the molecule, describing its core structure.
- **HeavyAtomMolWt**: Molecular weight of atoms in the molecule, excluding hydrogen atoms.
- **Kappa3**: Kappa-3 shape index of the molecule, describing its topology.
- **MaxAbsEStateIndex**: Maximum absolute E-State index of the molecule, representing its maximum charge state.
- **MinEStateIndex**: Minimum E-State index of the molecule, representing its minimum charge state.
- **NumHeteroatoms**: Number of heteroatoms (non-carbon atoms) in the molecule.
- **PEOE_VSA10, PEOE_VSA14, PEOE_VSA6, PEOE_VSA7, PEOE_VSA8**: PEOE (Partial Equalization of Orbital Electronegativity) - - VSA (Value Sum) 10, 14, 6, 7, 8, representing the partial charge surface area contributions of the molecule.
- **SMR_VSA10, SMR_VSA5**: SMR (Simple Molecular Representation) VSA 10, 5, representing the similarity model radius surface area contributions of the molecule.
- **SlogP_VSA3**: SlogP (Substructure-logP) VSA 3, representing the logarithm of the partition coefficient of the molecule.
- **VSA_EState9**: E-State-VSA (E-State Value Sum) 9, representing the electrotopological state contributions of the molecule related to its surface area.
- **fr_COO, fr_COO2**: Number of carboxyl groups in the molecule.

**Competition details and notebook objectives**<br>
1. This is a multi-label classification challenge to predict enzyme classes using the provided features. **GINI** is the metric for the challenge<br>
2. In this starter notebook, we start the assignment with a detailed EDA, feature plots, interaction effects, adversarial CV analysis and develop starter models to initiate the challenge. We will also incorporate other opinions and approaches as we move along the challenge.<br>
3. **Note:-** In this challenge, we could have a set of features being mapped to 2 classes, hence the name **multi-label classifier**. For multi-class, we need to exclusively classify features into 1 among many class options<br>

<a id="3"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > PREPROCESSING<br><div> 

In [ ]:
%time 

class Preprocessor():
    """
    This class aims to do the below-
    1. Read the datasets
    2. In this case, process the original data
    3. Check information and description
    4. Check unique values
    5. Collate starting features 
    6. Conjoin train-original data if requested based on Adversarial CV results
    """;
    
    def __init__(self):
        self.train    = pd.read_csv(CFG.path + f"train.csv", index_col = 'id');
        self.test     = pd.read_csv(CFG.path + f"test.csv", index_col = 'id');
        self.targets  = self.train.columns[self.train.columns.str.startswith("EC")].tolist();
        self.test_req = CFG.test_req;
        self.dtl_preproc_req = CFG.dtl_preproc_req;
        self.conjoin_orig_data = CFG.conjoin_orig_data;
        
        self.sub_fl   = pd.read_csv(CFG.path + f"sample_submission.csv");
        
        PrintColor(f"Data shape - train-test = {self.train.shape} {self.test.shape}");
        
        PrintColor(f"\nTrain set head", color = Fore.GREEN);
        display(self.train.head(5).style.format(precision = 3));
        PrintColor(f"\nTest set head", color = Fore.GREEN);
        display(self.test.head(5).style.format(precision = 3));
             
    def _ProcessOrig(self):
        o1 = pd.read_csv(CFG.orig_path + f"mixed_desc.csv", index_col = 'CIDs');
        o2 = pd.read_csv(CFG.orig_path + f"mixed_ecfp.csv", index_col = 'CIDs');
        o3 = pd.read_csv(CFG.orig_path + f"mixed_fcfp.csv", index_col = 'CIDs');
        
        self.original = pd.concat([o1.iloc[:, 0:-1], o2.iloc[:, 0:-1], o3], axis=1);
        _ = self.original[o3.columns[-1]].str.split('_', expand = True).astype(np.int8);
        _.columns = ['EC1', 'EC2', 'EC3', 'EC4', 'EC5', 'EC6'];
        self.original = pd.concat([self.original.drop(o3.columns[-1], axis=1), _], 
                                  axis = 1)[self.train.columns];
        del o1, o2, o3, _;
        
        # Resetting original data index:-
        self.original.index = range(len(self.original));
        self.original.index+= max(self.test.index) + 1;
        self.original.index.name = 'id';
        PrintColor(f"\nOriginal data shape -- {self.original.shape}");
        return self;
    
    def _SampleData(self):
        if self.test_req == "Y":
            PrintColor(f"---> We are testing the code with 5% data sample", color = Fore.RED);
            self.train     = self.train.groupby(CFG.target).sample(frac = 0.05);
            self.original  = self.original.groupby(CFG.target).sample(frac = 0.05);
            self.test      = self.test.sample(frac = 0.05);
            self.sub_fl    = self.sub_fl.loc[self.sub_fl.id.isin(self.test.index)];
        
        return self;
    
    def _AddSourceCol(self):
        self.train['Source'] = "Competition";
        self.test['Source']  = "Competition";
        self.original['Source'] = 'Original';
        
        self.strt_ftre = self.test.columns;
        return self;
    
    def _CollateInfoDesc(self):
        if self.dtl_preproc_req == "Y":
            PrintColor(f"\n{'-'*20} Information and description {'-'*20}\n", color = Fore.MAGENTA);

            # Creating dataset information and description:
            for lbl, df in {'Train': self.train, 'Test': self.test, 'Original': self.original}.items():
                PrintColor(f"\n{lbl} description\n");
                display(df.describe(percentiles= [0.05, 0.25, 0.50, 0.75, 0.9, 0.95, 0.99]).\
                        transpose().\
                        drop(columns = ['count'], errors = 'ignore').\
                        drop([CFG.target], axis=0, errors = 'ignore').\
                        style.format(formatter = '{:,.2f}').\
                        background_gradient(cmap = 'Blues')
                       );

                PrintColor(f"\n{lbl} information\n");
                display(df.info());
                collect();
        return self;
    
    def _CollateUnqNorm(self):
        if self.dtl_preproc_req == "Y": 
            
            PrintColor(f"\n{'-'*20} Unique values and normality tests {'-'*20}\n", color = Fore.MAGENTA);

            # Dislaying the unique values across train-test-original:-
            PrintColor(f"\nUnique values\n");
            _ = pd.concat([self.train[self.strt_ftre].nunique(), 
                           self.test[self.strt_ftre].nunique(), 
                           self.original[self.strt_ftre].nunique()], 
                          axis=1);
            _.columns = ['Train', 'Test', 'Original'];

            display(_.T.style.background_gradient(cmap = 'Blues', axis=1).\
                    format(formatter = '{:,.0f}')
                   );

            # Normality check:-
            cols = list(self.strt_ftre[0:-1]);
            
            for lbl, test_lbl in {"Shapiro": shapiro, "NormalTest": normaltest}.items():
                PrintColor(f"\n{lbl} normality analysis\n");
                pprint({col: [np.round(test_lbl(self.train[col]).pvalue,decimals = 4), 
                              np.round(test_lbl(self.test[col]).pvalue,4) if col != CFG.target else np.NaN,
                              np.round(test_lbl(self.original[col]).pvalue,4)] for col in cols
                       }, indent = 5, width = 100, depth = 2, compact= True);   

        return self;
       
    def DoPreprocessing(self):
        self._ProcessOrig();
        self._SampleData();
        self._AddSourceCol();
        self._CollateInfoDesc();
        self._CollateUnqNorm();
        
        return self; 
        
    def ConjoinTrainOrig(self):
        if self.conjoin_orig_data == "Y":
            PrintColor(f"Train shape before conjoining with original = {self.train.shape}");
            train = pd.concat([self.train, self.original], axis=0, ignore_index = True);
            PrintColor(f"Train shape after conjoining with original= {train.shape}");
            
            train = train.drop_duplicates();
            PrintColor(f"Train shape after de-duping = {train.shape}");
            
            train.index = range(len(train));
            train.index.name = 'id';
        
        else:
            PrintColor(f"We are using the competition training data only");
            train = self.train;
        return train;
          
collect();
print();

In [ ]:
%%time 

pp = Preprocessor();
pp.DoPreprocessing();

print();
collect();


## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > INFERENCES<br> <div>

<div style= "font-family: Cambria; letter-spacing: 0px; color:#000000; font-size:110%; text-align:left;padding:3.0px; background: #f2f2f2" >
1. All the columns are numerical<br>
2. We do not have any nulls in the data<br>
3. All columns are non-normal<br>
4. The synthetic data is nearly 24.6 times the original data, creating a potential quasi-duplicate row issue. Duplicate handling could be a key challenge in this case<br>
5. This is a multi-label classifier with 6 targets, we need only 2 for the challenge, are the rest superfluous?<br>
6. fr_coo and fr_coo2 appear to be categorical columns<br>
</div>

<a id="4"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > ADVERSARIAL CV<br><div>

In [ ]:
%%time

# Performing adversarial CV between the 2 specified datasets:-
def Do_AdvCV(df1:pd.DataFrame, df2:pd.DataFrame, source1:str, source2:str):
    "This function performs an adversarial CV between the 2 provided datasets if needed by the user";
    
    # Adversarial CV per column:-
    ftre = pp.test.select_dtypes(include = np.number).\
    drop(columns = ['id', "Source"], errors = 'ignore').columns;
    adv_cv = {};

    for col in ftre:
        shuffle_state = np.random.randint(low = 10, high = 100, size= 1);

        full_df = \
        pd.concat([df1[[col]].assign(Source = source1), df2[[col]].assign(Source = source2)], 
                  axis=0, ignore_index = True).\
        sample(frac = 1.00, random_state = shuffle_state);

        full_df = full_df.assign(Source_Nb = full_df['Source'].eq(source2).astype(np.int8));

        # Checking for adversarial CV:-
        model = LGBMClassifier(random_state = CFG.state, max_depth = 6, learning_rate = 0.05);
        cv    = all_cv['SKF'];
        score = np.mean(cross_val_score(model, 
                                        full_df[[col]], 
                                        full_df.Source_Nb, 
                                        scoring= 'roc_auc', 
                                        cv     = cv)
                       );
        adv_cv.update({col: round(score, 4)});
        collect();
    
    del ftre;
    collect();
    
    fig, ax = plt.subplots(1,1,figsize = (12, 5));
    pd.Series(adv_cv).plot.bar(color = 'tab:blue', ax = ax);
    ax.axhline(y = 0.60, color = 'red', linewidth = 2.75);
    ax.grid(**CFG.grid_specs); 
    plt.yticks(np.arange(0.0, 0.81, 0.05));
    plt.show();
    
# Implementing the adversarial CV:-
if CFG.adv_cv_req == "Y":
    PrintColor(f"\n---------- Adversarial CV - Train vs Original ----------\n", 
               color = Fore.MAGENTA);
    Do_AdvCV(df1 = pp.train, df2 = pp.original, source1 = 'Train', source2 = 'Original');
    
    PrintColor(f"\n---------- Adversarial CV - Train vs Test ----------\n", 
               color = Fore.MAGENTA);
    Do_AdvCV(df1 = pp.train, df2 = pp.test, source1 = 'Train', source2 = 'Test');
    
    PrintColor(f"\n---------- Adversarial CV - Original vs Test ----------\n", 
               color = Fore.MAGENTA);
    Do_AdvCV(df1 = pp.original, df2 = pp.test, source1 = 'Original', source2 = 'Test');   
    
if CFG.adv_cv_req == "N":
    PrintColor(f"\nAdversarial CV is not needed\n", color = Fore.RED);
    
collect();
print();

In [ ]:
%%time 

print();
train, test, strt_ftre = pp.ConjoinTrainOrig(), pp.test.copy(deep = True), deepcopy(pp.strt_ftre);
cat_cols  = ['fr_COO', 'fr_COO2'];
cont_cols = [col for col in strt_ftre if col not in cat_cols + ['Source']];

PrintColor(f"\nCategory columns\n");
display(cat_cols);
PrintColor(f"\nContinuous columns\n");
display(np.array(cont_cols));
PrintColor(f"\nAll columns\n");
display(strt_ftre);

print();
collect();

## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > INFERENCES<br><div>

<div style= "font-family: Cambria; letter-spacing: 0px; color:#000000; font-size:110%; text-align:left;padding:3.0px; background: #f2f2f2" >
1. Train-test belong to the same distribution, we can perhaps rely on the CV score<br>
2. We need to further check the train-original distribution further, adversarial validation results indicate that we can use the original dataset<br>
</div>

<a id="5"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > VISUALS AND EDA <br><div> 
 

<a id="5.2"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > TARGET PLOT<br><div>

In [ ]:
%%time 

if CFG.ftre_plots_req == "Y":
    
    for tgt in CFG.target:
        fig, axes = plt.subplots(1,2, figsize = (12, 5), sharey = True, gridspec_kw = {'wspace': 0.2});

        for i, df in tqdm(enumerate([pp.train, pp.original]), "Target balance ---> "):
            ax= axes[i];
            a = df[tgt].value_counts(normalize = True);
            _ = ax.pie(x = a , labels = a.index.values, 
                       explode      = [0.0, 0.3], 
                       startangle   = 40, 
                       shadow       = True, 
                       colors       = ['#3377ff', '#66ffff'], 
                       textprops    = {'fontsize': 7, 'fontweight': 'bold', 'color': 'black'},
                       pctdistance  = 0.60, 
                       autopct = '%1.1f%%'
                      );
            df_name = 'Train' if i == 0 else "Original";
            _ = ax.set_title(f"\n{df_name} data- {tgt}\n", **CFG.title_specs);

        plt.tight_layout();
        plt.show();
        
        
    
collect();
print();

In [ ]:
%%time 

# Assessing target interactions:-
if CFG.ftre_plots_req == "Y":
    fig, axes = plt.subplots(1,2, figsize = (12, 4), gridspec_kw = {'wspace': 0.2});
    
    for i, (lbl, df) in enumerate({"Train": pp.train, "Original": pp.original}.items()):
        ax = axes[i];
        c = ['#3377ff', '#6699cc'];
        df.groupby(CFG.target).size().plot.bar(ax = ax, color = c[i]);
        ax.set_title(f"Target interaction - {lbl} set", **CFG.title_specs);
        ax.set(xlabel = "");
        
    plt.tight_layout();
    plt.show()
        

<a id="5.4"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > CATEGORY COLUMN PLOTS<br><div>

In [ ]:
%%time

if CFG.ftre_plots_req == "Y":
    fig, axes = plt.subplots(len(cat_cols), 3, figsize = (20, len(cat_cols)* 4.5), 
                             gridspec_kw = {'wspace': 0.2, 'hspace': 0.3});

    for i, col in enumerate(cat_cols):
        ax = axes[i, 0];
        a = pp.train[col].value_counts(normalize = True);
        a.sort_index().plot.barh(ax = ax, color = '#007399');
        ax.set_title(f"{col}_Train", **CFG.title_specs);
        ax.set_xticks(np.arange(0.0, 0.7, 0.03), 
                      labels = np.round(np.arange(0.0, 0.7, 0.03),2), 
                      rotation = 90);
        del a;

        ax = axes[i, 1];
        a = pp.test[col].value_counts(normalize = True);
        a.sort_index().plot.barh(ax = ax, color = '#0088cc');
        ax.set_title(f"{col}_Test", **CFG.title_specs);
        ax.set_xticks(np.arange(0.0, 0.7, 0.03), 
              labels = np.round(np.arange(0.0, 0.7, 0.03),2), 
              rotation = 90);
        del a;
        
        ax = axes[i, 2];
        a = pp.original[col].value_counts(normalize = True);
        a.sort_index().plot.barh(ax = ax, color = '#0047b3');
        ax.set_title(f"{col}_Original", **CFG.title_specs);
        ax.set_xticks(np.arange(0.0, 0.7, 0.03), 
              labels = np.round(np.arange(0.0, 0.7, 0.03),2), 
              rotation = 90);
        del a;       

    plt.tight_layout();
    plt.show();
    
print();
collect();

<a id="5.5"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > CONTINUOUS COLUMN PLOTS<br><div>

In [ ]:
%%time 

if CFG.ftre_plots_req == "Y":
    df = pd.concat([pp.train[cont_cols].assign(Source = 'Train'), 
                    pp.test[cont_cols].assign(Source = 'Test'),
                    pp.original[cont_cols].assign(Source = "Original")
                   ], 
                   axis=0, ignore_index = True
                  );
    
    fig, axes = plt.subplots(len(cont_cols), 4 ,figsize = (16, len(cont_cols) * 4.2), 
                             gridspec_kw = {'hspace': 0.35, 'wspace': 0.3, 'width_ratios': [0.80, 0.20, 0.20, 0.20]});
    
    for i,col in enumerate(cont_cols):
        ax = axes[i,0];
        sns.kdeplot(data = df[[col, 'Source']], x = col, hue = 'Source', 
                    palette = ['#0039e6', '#ff5500', '#00b300'], 
                    ax = ax, linewidth = 2.1
                   );
        ax.set_title(f"\n{col}", **CFG.title_specs);
        ax.grid(**CFG.grid_specs);
        ax.set(xlabel = '', ylabel = '');
        
        ax = axes[i,1];
        sns.boxplot(data = df.loc[df.Source == 'Train', [col]], y = col, width = 0.25,
                    color = '#33ccff', saturation = 0.90, linewidth = 0.90, 
                    fliersize= 2.25,
                    ax = ax);
        ax.set(xlabel = '', ylabel = '');
        ax.set_title(f"Train", **CFG.title_specs);
        
        ax = axes[i,2];
        sns.boxplot(data = df.loc[df.Source == 'Test', [col]], y = col, width = 0.25, fliersize= 2.25,
                    color = '#80ffff', saturation = 0.6, linewidth = 0.90, 
                    ax = ax); 
        ax.set(xlabel = '', ylabel = '');
        ax.set_title(f"Test", **CFG.title_specs);
        
        ax = axes[i,3];
        sns.boxplot(data = df.loc[df.Source == 'Original', [col]], y = col, width = 0.25, fliersize= 2.25,
                    color = '#99ddff', saturation = 0.6, linewidth = 0.90, 
                    ax = ax); 
        ax.set(xlabel = '', ylabel = '');
        ax.set_title(f"Original", **CFG.title_specs);
              
    plt.suptitle(f"\nDistribution analysis- continuous columns\n", **CFG.title_specs, 
                 y = 0.89, x = 0.57);
    plt.tight_layout();
    plt.show();
    
print();
collect();

<a id="5.7"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > FEATURE INTERACTION AND UNIVARIATE RELATIONS<br><div>
    
We aim to do the below herewith<br>
1. Correlation<br>
2. Mutual information<br>

In [ ]:
%%time 

def MakeCorrPlot(df: pd.DataFrame, data_label:str, figsize = (30, 9)):
    """
    This function develops the correlation plots for the given dataset
    """;
    
    fig, axes = plt.subplots(1,2, figsize = figsize, gridspec_kw = {'hspace': 0.2, 'wspace': 0.1},
                             sharey = True
                            );
    
    for i, method in enumerate(['pearson', 'spearman']):
        corr_ = df.drop(columns = ['id', 'Source'], errors = 'ignore').corr(method = method);
        ax = axes[i];
        sns.heatmap(data = corr_,  
                    annot= True,
                    fmt= '.2f', 
                    cmap = 'Blues',
                    annot_kws= {'fontweight': 'bold','fontsize': 6.75}, 
                    linewidths= 1.5, 
                    linecolor='white', 
                    cbar= False, 
                    mask= np.triu(np.ones_like(corr_)),
                    ax= ax
                   );
        ax.set_title(f"\n{method.capitalize()} correlation- {data_label}\n", **CFG.title_specs);
        
    collect();
    print();

# Implementing correlation analysis:-
for lbl, df in {"Train": pp.train, "Test": pp.test, "Original": pp.original}.items():
    MakeCorrPlot(df = df, data_label = lbl, figsize = (38, 13));

print();
collect();

In [ ]:
%%time 

for tgt in CFG.target:
    MutInfoSum = {};
    for i, df in enumerate([pp.train, pp.original]):
        MutInfoSum.update({'Train' if i == 0 else 'Original': 
                           mutual_info_classif(df[strt_ftre[0:-1]], df[tgt], random_state = CFG.state)
                          });

    MutInfoSum = pd.DataFrame(MutInfoSum, index = strt_ftre[0:-1]);

    fig, axes = plt.subplots(1,2, figsize = (28, 6), gridspec_kw = {'wspace': 0.2});
    colors = ['#4080bf', '#3377ff'];
    for i in range(2):
        MutInfoSum.iloc[:, i].plot.bar(ax = axes[i], color = colors[i]);
        axes[i].set_title(f"{MutInfoSum.columns[i]} - Mutual Information -- {tgt}", **CFG.title_specs);

    plt.tight_layout();
    plt.show();
print();
collect();

<a id="5.9"></a>
## <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0059b3; border-bottom: 8px solid #e6e6e6" > INFERENCES<br> <div>

<div style= "font-family: Cambria; letter-spacing: 0px; color:#000000; font-size:110%; text-align:left;padding:3.0px; background: #f2f2f2" >
1. Feature selection is a very important part of the assignment. We have lots of features and feature selection will be a differentiator<br>
2. Almost all features have outliers. Outlier handling will be another differentiator in this challenge<br>
3. All features are non-normal, certain features like Fpdensity and Kappa3 need to be assessed in the next runs<br>
4. Should be dump the other EC columns (EC3-EC6)? I am sure we will extract valuable information from these columns<br>
5. Columns are highly correlated. Dimensionality reduction will surely help here<br>
6. Target interaction is common here, we have a sizable base with 1s in both EC1 and EC2<br>
</div>

<a id="6"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > DATA TRANSFORMS <br><div> 
    
This section aims at creating secondary features, scaling and if necessary, conjoining the competition training and original data tables<br>
Currently we will leave this and add them later<

In [ ]:
%%time 

# Data transforms:-
class Xformer(TransformerMixin, BaseEstimator):
    """
    This class is used to create secondary features from the existing data
    """;
    
    def __init__(self): pass
    
    def fit(self, X, y= None, **params):
        self.ip_cols = X.columns;
        return self;
    
    def transform(self, X, y= None, **params):       
        global strt_ftre;
        df    = X.copy();      
      
        if CFG.sec_ftre_req == "Y":
            pass
        if CFG.sec_ftre_req != "Y": 
            PrintColor(f"Secondary features are not required", color = Fore.RED);    
        
        self.op_cols = df.columns;  
        return df;
    
    def get_feature_names_in(self, X, y=None, **params): 
        return self.ip_cols;    
    
    def get_feature_names_out(self, X, y=None, **params): 
        return self.op_cols;
     
# Scaling:-
class Scaler(TransformerMixin, BaseEstimator):
    """
    This class aims to create scaling for the provided dataset
    """;
    
    def __init__(self, scl_method: str, scale_req: str, scl_cols):
        self.scl_method = scl_method;
        self.scale_req  = scale_req;
        self.scl_cols   = scl_cols;
        
    def fit(self,X, y=None, **params):
        "This function calculates the train-set parameters for scaling";
        
        self.params          = X[self.scl_cols].describe(percentiles = [0.25, 0.50, 0.75]).drop(['count'], axis=0).T;
        self.params['iqr']   = self.params['75%'] - self.params['25%'];
        self.params['range'] = self.params['max'] - self.params['min'];
        
        return self;
    
    def transform(self,X, y=None, **params):  
        "This function transform the relevant scaling columns";
        
        df = X.copy();
        if self.scale_req == "Y":
            if CFG.scl_method == "Z":
                df[self.scl_cols] = (df[self.scl_cols].values - self.params['mean'].values) / self.params['std'].values;
            elif CFG.scl_method == "Robust":
                df[self.scl_cols] = (df[self.scl_cols].values - self.params['50%'].values) / self.params['iqr'].values;
            elif CFG.scl_method == "MinMax":
                df[self.scl_cols] = (df[self.scl_cols].values - self.params['min'].values) / self.params['range'].values;
        else:
            PrintColor(f"Scaling is not needed", color = Fore.RED);
    
        return df;
    
collect();
print();

In [ ]:
strt_ftre

In [ ]:
%%time 

PrintColor(f"\n{'-'* 20} Data transforms and encoding {'-'* 20}", color = Fore.MAGENTA);

# Data transforms:--
Xtrain, Ytrain = train[strt_ftre], train[CFG.target];

pca = make_pipeline(*[Scaler(scl_method = CFG.scl_method, scale_req = "Y", scl_cols = cont_cols), 
                      PCA(random_state = CFG.state, n_components= CFG.ncomp)
                     ]
                   );
xform = \
make_pipeline(*[ColumnTransformer([('D', pca, cont_cols), 
                                   ('T', Xformer(), strt_ftre)
                                  ], verbose_feature_names_out= False, remainder= 'passthrough')]
             );
print();
display(xform);

xform.fit(Xtrain, Ytrain[CFG.target[0]]);
Xtrain = xform.transform(Xtrain);
Xtest  = xform.transform(pp.test);

# Adjusting the indices after transforms:-
Xtrain.index = range(len(Xtrain));
Ytrain.index = Xtrain.index;

print();
collect();

In [ ]:
%%time 

# Displaying the transformed data descriptions for infinite/ null values:-
PrintColor(f"\n---- Transformed data description for distribution analysis ----\n",
          color = Fore.MAGENTA);

PrintColor(f"\nTrain data\n");
display(Xtrain.describe(percentiles = [0.05, 0.25, 0.50, 0.75, 0.9, 0.95]).T.\
        drop(columns = ['count']).\
        style.format(formatter = '{:,.2f}').\
        background_gradient(cmap = 'Blues')
       );

PrintColor(f"\nTest data\n");
display(Xtest.describe(percentiles = [0.05, 0.25, 0.50, 0.75, 0.9, 0.95]).T.\
        drop(columns = ['count']).\
        style.format(formatter = '{:,.2f}').\
        background_gradient(cmap = 'Blues')
       );


<a id="7"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > MODEL TRAINING <br><div> 
    
We commence our model assignment with a simple ensemble of tree-based and linear models and then shall proceed with the next steps<br>
  
**Note**-<br>
GINI metric is a rank based metric that does not necessitate the usage of classifiers only. Regressors could be used to good effect appropos to their contribution to the CV score. <br>
**We will make 2 classifiers, one for each target and then execute the process**<br>

In [ ]:
%%time 

# Initializing model I-O:-

Mdl_Master = \
{'CBR': CatBoostRegressor(**{'task_type'           : "GPU" if CFG.gpu_switch == "ON" else "CPU",
                             'loss_function'       : 'RMSE',
                             'eval_metric'         : 'RMSE',
                             'bagging_temperature' : 0.45,
                             'colsample_bylevel'   : 0.75,
                             'iterations'          : 4500,
                             'learning_rate'       : 0.085,
                             'od_wait'             : 40,
                             'max_depth'           : 7,
                             'l2_leaf_reg'         : 0.75,
                             'min_data_in_leaf'    : 35,
                             'random_strength'     : 0.2, 
                             'max_bin'             : 256,
                             'verbose'             : 0,
                           }
                        ),
 
 'CBC': CatBoostClassifier(**{'task_type'           : "GPU" if CFG.gpu_switch == "ON" else "CPU",
                              'objective'           : 'Logloss',
                              'loss_function'       : 'Logloss',
                              'eval_metric'         : 'AUC',
                              'bagging_temperature' : 0.425,
                              'colsample_bylevel'   : 0.75,
                              'iterations'          : 4_000,
                              'learning_rate'       : 0.025,
                              'od_wait'             : 32,
                              'max_depth'           : 6,
                              'l2_leaf_reg'         : 0.45,
                              'min_data_in_leaf'    : 28,
                              'random_strength'     : 0.15, 
                              'max_bin'             : 200,
                              'verbose'             : 0,
                           }
                         ), 

 'LGBMR': LGBMRegressor(**{'device'            : "gpu" if CFG.gpu_switch == "ON" else "cpu",
                           'objective'         : 'regression',
                           'metric'            : 'rmse',
                           'boosting_type'     : 'gbdt',
                           'random_state'      : CFG.state,
                           'colsample_bytree'  : 0.67,
                           'feature_fraction'  : 0.70,
                           'learning_rate'     : 0.06,
                           'max_depth'         : 8,
                           'n_estimators'      : 5000,
                           'num_leaves'        : 120,                    
                           'reg_alpha'         : 1.25,
                           'reg_lambda'        : 3.5,
                           'verbose'           : -1,
                         }
                      ),
 
  'LGBMC': LGBMClassifier(**{'device'            : "gpu" if CFG.gpu_switch == "ON" else "cpu",
                             'objective'         : 'binary',
                             'metric'            : 'auc',
                             'boosting_type'     : 'gbdt',
                             'random_state'      : CFG.state,
                             'colsample_bytree'  : 0.675,
                             'subsample'         : 0.925,
                             'learning_rate'     : 0.025,
                             'max_depth'         : 9,
                             'n_estimators'      : 4000,
                             'num_leaves'        : 90,                    
                             'reg_alpha'         : 0.0001,
                             'reg_lambda'        : 1.5,
                             'verbose'           : -1,
                         }
                      ),

 'XGBR': XGBRegressor(**{'objective'          : 'reg:squarederror',
                         'eval_metric'        : 'rmse',
                         'random_state'       : CFG.state,
                         'tree_method'        : "gpu_hist" if CFG.gpu_switch == "ON" else "hist",
                         'colsample_bytree'   : 0.75,
                         'learning_rate'      : 0.0125,
                         'max_depth'          : 8,
                         'n_estimators'       : 5000,                         
                         'reg_alpha'          : 1.25,
                         'reg_lambda'         : 1e-05,
                         'min_child_weight'   : 40,
                        }
                     ),
 
  'XGBC': XGBClassifier(**{'tree_method'        : "gpu_hist" if CFG.gpu_switch == "ON" else "hist",
                           'objective'          : 'binary:logistic',
                           'eval_metric'        : 'auc',
                           'random_state'       : CFG.state,
                           'colsample_bytree'   : 0.25,
                           'learning_rate'      : 0.018,
                           'max_depth'          : 8,
                           'n_estimators'       : 4000,                         
                           'reg_alpha'          : 0.0001,
                           'reg_lambda'         : 2.25,
                           'min_child_weight'   : 50,
                        }
                       ),
 
  'HGBC': HGBC(learning_rate    = 0.045,
               max_iter         = 2000,
               max_depth        = 8,
               min_samples_leaf = 32,
               l2_regularization= 1.25,
               max_bins         = 200,
               n_iter_no_change = 50,
               random_state     = CFG.state,
              ),
 
  'RFC' : RFC(n_estimators     = 250,
              criterion        = 'gini',
              max_depth        = 6,
              min_samples_leaf = 35,
              max_features     = "log2",
              bootstrap        = True,
              oob_score        = True,
              n_jobs           = -1,
              random_state     = CFG.state,
              verbose          = 0,
             ),
 
  'ETC' : ETC(n_estimators     = 350,
              criterion        = 'gini',
              max_depth        = 7,
              min_samples_leaf = 38,
              max_features     = "log2",
              bootstrap        = True,
              oob_score        = True,
              n_jobs           = -1,
              random_state     = CFG.state,
              verbose          = 0,
             ),
 
  'LC' : LC(penalty='l2',
            tol=0.0001,
            C= 5.3,
            random_state= CFG.state,
           )
};

print();
collect();

In [ ]:
%%time

# Selecting relevant columns for the train and test sets:-
sel_cols = ['pca0', 'pca1', 'pca2', 
            'BertzCT', 'Chi1', 'Chi1n', 'Chi1v', 'Chi2n','Chi2v', 'Chi3v', 'Chi4n',
            'EState_VSA1', 'EState_VSA2', 'ExactMolWt',
            'FpDensityMorgan1', 'FpDensityMorgan2', 'FpDensityMorgan3',
            'HallKierAlpha', 'HeavyAtomMolWt', 'Kappa3', 'MaxAbsEStateIndex',
            'MinEStateIndex', 'NumHeteroatoms', 
            'PEOE_VSA10', 'PEOE_VSA14', 'PEOE_VSA6', 'PEOE_VSA7','PEOE_VSA8', 
            'SMR_VSA10', 'SMR_VSA5','SlogP_VSA3', 'VSA_EState9',
            'fr_COO', 'fr_COO2',
            'Source'
           ];
print(); 

try: 
    Xtrain, Xtest = Xtrain[sel_cols], Xtest[sel_cols];
    pprint(Xtest.columns, depth = 1, width = 10, indent = 5);
except: 
    PrintColor(f"\n---> Check the columns selected\n---> Selected columns-", color = Fore.RED);
    pprint(Xtest.columns, depth = 1, width = 10, indent = 5);
        
# Initializing output tables for the models:-
methods   = [col for col in Mdl_Master.keys() if col.endswith("C")];
OOF_Preds = pd.DataFrame();
Mdl_Preds = pd.DataFrame(index = pp.sub_fl['id']);
FtreImp   = pd.DataFrame(index = Xtrain.drop(columns = ['Source'], errors = 'ignore').columns);
Scores    = pd.DataFrame(columns = methods);

PrintColor(f"\n---> Selected model options- ");
pprint(methods, depth = 1, width = 100, indent = 5);

print();
collect();

In [ ]:
%%time 

def TrainMdl(method:str, ytrain):
    
    global Mdl_Master, Mdl_Preds, OOF_Preds, all_cv, FtreImp, Xtrain; 
    
    model     = Mdl_Master.get(method); 
    cols_drop = ['id', 'Source', 'Label'];
    scl_cols  = [col for col in Xtrain.columns if col not in cols_drop + ['pca0', 'pca1', 'pca2']];
    cv        = all_cv.get(CFG.mdlcv_mthd);
    Xt        = Xtest.copy(deep = True);
    
    if CFG.scale_req == "N" and method.upper() in ['RIDGE', 'LASSO', 'SVR', "SVC", "LC"]:
        X, y        = Xtrain.copy(deep = True), ytrain.copy(deep = True);
        scaler      = all_scalers[CFG.scl_method];
        X[scl_cols] = scaler.fit_transform(X[scl_cols]);
        Xt[scl_cols]= scaler.transform(Xt[scl_cols]);
        PrintColor(f"--> Scaling the data for {method} model");

    if CFG.use_orig_allfolds == "Y":
        X    = Xtrain.query("Source == 'Competition'");
        y    = ytrain.loc[ytrain.index.isin(X.index)]; 
        Orig = pd.concat([Xtrain, ytrain], axis=1).query("Source == 'Original'");
        
    elif CFG.use_orig_allfolds != "Y":
        X,y = Xtrain.copy(deep = True), ytrain.copy(deep = True);
                
    # Initializing I-O for the given seed:-        
    test_preds = 0;
    oof_preds  = pd.DataFrame(); 
    scores     = [];
    ftreimp    = 0;
          
    for fold_nb, (train_idx, dev_idx) in enumerate(cv.split(X, y)): 
        Xtr  = X.iloc[train_idx].drop(columns = cols_drop, errors = 'ignore');   
        Xdev = X.iloc[dev_idx].loc[X.Source == "Competition"].\
        drop(columns = cols_drop, errors = 'ignore'); 
        ytr  = y.loc[y.index.isin(Xtr.index)];
        ydev = y.loc[y.index.isin(Xdev.index)];

        if CFG.use_orig_allfolds == "Y":
            Xtr = pd.concat([Xtr, Orig.drop(columns = [CFG.target, 'Source'], errors = 'ignore')], 
                            axis = 0, ignore_index = True);
            ytr = pd.concat([ytr, Orig[CFG.target]], axis = 0, ignore_index = True);
            
        # Fitting the model:- 
        if method in ['CBR', 'CBC']:    
            model.fit(Xtr, ytr, 
                      eval_set = [(Xdev, ydev)], 
                      verbose = 0,
                      early_stopping_rounds = CFG.nbrnd_erly_stp,
                      cat_features = cat_cols,
                     ); 
            
        elif method in ['LGBMR', 'LGBMC']: 
            model.fit(Xtr, ytr, 
                      eval_set = [(Xdev, ydev)], 
                      verbose = 0,
                      early_stopping_rounds = CFG.nbrnd_erly_stp,
                      categorical_feature = cat_cols,
                     );
            
        elif method in ['XGBR', 'XGBC']:
            model.fit(Xtr, ytr, 
                      eval_set = [(Xdev, ydev)], 
                      verbose = 0,
                      early_stopping_rounds = CFG.nbrnd_erly_stp,
                     );            
           
        else: 
            model.fit(Xtr, ytr); 
            
        # Collecting predictions and scores and post-processing OOF based on model method:-
        if method.upper().endswith('R'):
            dev_preds = PostProcessPred(model.predict(Xdev), 
                                        post_process= CFG.pstprcs_train);
            test_preds = test_preds + \
            PostProcessPred(model.predict(Xt.drop(columns = cols_drop, errors = 'ignore')),
                            post_process= CFG.pstprcs_train); 
        else:
            dev_preds = model.predict_proba(Xdev)[:,1];
            test_preds = test_preds + \
            model.predict_proba(Xt.drop(columns = cols_drop, errors = 'ignore'))[:,1];            
        
        score = ScoreMetric(ydev.values.flatten(), dev_preds);
        scores.append(score); 
 
        Scores.loc[f"{tgt}_Fold{fold_nb}", method] = np.round(score, decimals= 6);
        oof_preds = pd.concat([oof_preds,
                               pd.DataFrame(index   = Xdev.index, 
                                            data    = dev_preds,
                                            columns = [method])
                              ],axis=0, ignore_index= False
                             );  
    
        oof_preds = pd.DataFrame(oof_preds.groupby(level = 0)[method].mean());
        oof_preds.columns = [method];
        
        try: ftreimp += model.feature_importances_;
        except: ftreimp = 0;
             
    
    OOF_Preds[f'{method}_{tgt}'] = PostProcessPred(oof_preds.values.flatten(), CFG.pstprcs_train);
    
    if CFG.mdlcv_mthd in ['KF', 'SKF']:
        Mdl_Preds[f'{method}_{tgt}'] = test_preds.flatten()/ CFG.n_splits; 
        FtreImp[f'{method}_{tgt}']   = ftreimp / CFG.n_splits;
    else:
        Mdl_Preds[f'{method}_{tgt}'] = test_preds.flatten()/ (CFG.n_splits * CFG.n_repeats); 
        FtreImp[f'{method}_{tgt}']   = ftreimp / (CFG.n_splits * CFG.n_repeats);
    
    collect(); 
    
collect();
print();

In [ ]:
%%time 

# Implementing the ML models:-
if CFG.ML == "Y": 
    for tgt in CFG.target:
        PrintColor(f"{'-'* 40} {tgt} {'-'* 40}", color = Fore.MAGENTA);
        for method in tqdm(methods, "ML models----"): 
            TrainMdl(method, ytrain = Ytrain[tgt]);
            
        PrintColor(f"\n{'-' * 20} Mean CV scores till {tgt} {'-' * 20}\n", color = Fore.MAGENTA);
        display(pd.concat([Scores.mean(axis = 0), Scores.std(axis = 0)], axis=1).\
                rename(columns = {0: 'Mean', 1: 'Std'}).T.\
                style.format(precision = 6).\
                background_gradient(cmap = 'Pastel1', axis=1)
               );    
else:
    PrintColor(f"\nML models are not needed\n", color = Fore.RED);
    
print();
collect();

In [ ]:
%%time

# Analysing the model results and feature importances and calibration curves:-
if CFG.ML == "Y":
    for tgt in CFG.target:
        ytrain = Ytrain[tgt];
        
        PrintColor(f"\n{'=' * 150}\n", color = Fore.MAGENTA);
        
        fig, axes = plt.subplots(len(methods), 2, figsize = (25, len(methods) * 7.5),
                                 gridspec_kw = {'hspace': 0.2, 'wspace': 0.2}, 
                                 width_ratios= [0.7, 0.3],
                                );

        for i, col in enumerate(methods):
            ax = axes[i,0];
            FtreImp[f"{col}_{tgt}"].plot.barh(ax = ax, color = '#0073e6');
            ax.set_title(f"{col}_{tgt} Importances", **CFG.title_specs);
            ax.set(xlabel = '', ylabel = '');

            ax = axes[i,1];
            Clb.from_predictions(ytrain[0:len(OOF_Preds)], OOF_Preds[f"{col}_{tgt}"], 
                                 n_bins= 20, ref_line = True,
                                 **{'color': '#0073e6', 'linewidth': 1.2, 
                                    'markersize': 3.75, 'marker': 'o', 'markerfacecolor': '#cc7a00'},
                                 ax = ax
                                );
            ax.set_title(f"{col} {tgt} Calibration", **CFG.title_specs);
            ax.set(xlabel = '', ylabel = '',);
            ax.set_yticks(np.arange(0,1.01, 0.05), labels = np.round(np.arange(0,1.01, 0.05), 2), fontsize = 7.0);
            ax.set_xticks(np.arange(0,1.01, 0.05), 
                          labels = np.round(np.arange(0,1.01, 0.05), 2), 
                          fontsize = 7.0, 
                          rotation = 90
                         );
            ax.legend('');

        plt.tight_layout();
        plt.show();
    
collect();
print();

<a id="8"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:black; font-size:120%; text-align:left;padding:3.0px; background: #cceeff; border-bottom: 8px solid #004466" > ENSEMBLE AND SUBMISSION<br> <div> 
   

In [ ]:
%%time 

if CFG.ensemble_req == "Y":
    def Objective(trial):
        "This function defines the objective for the optuna ensemble using variable models";

        global OOF_Preds, all_cv, ytrain, methods, tgt, cols;

        # Define the weights for the predictions from each model:-
        weights  = [trial.suggest_float(f"M{n}", 0.0001, 0.9999, step = 0.001) \
                    for n in range(len(cols))
                   ];

        # Calculating the CV-score for the weighted predictions on the competition data only:-
        scores = [];  
        cv     = all_cv[CFG.enscv_mthd];
        X,y    = OOF_Preds[cols], ytrain[0: len(OOF_Preds)];

        for fold_nb, (train_idx, dev_idx) in enumerate(cv.split(X,y)):
            Xtr, Xdev = X.iloc[train_idx], X.iloc[dev_idx];
            ytr, ydev = y.loc[Xtr.index],  y.loc[Xdev.index];
            scores.append(ScoreMetric(ydev, np.average(Xdev, axis=1, weights = weights)));

        collect();
        clear_output();
        return np.mean(scores);
    
clear_output();
print();
collect();

In [ ]:
%%time 

if CFG.ensemble_req == "Y":
    sub_fl      = pp.sub_fl.copy(deep = True);
    ens_weights = {};
    ens_score   = {};
    
    for tgt in tqdm(CFG.target, f"Ensemble -"):
        PrintColor(f"\n{'-' * 20} Optuna Ensemble for {tgt} {'-' * 20}\n", color = Fore.MAGENTA); 
        ytrain = Ytrain[tgt];
        cols   = OOF_Preds.columns[OOF_Preds.columns.str.endswith(tgt)].tolist();

        study = optuna.create_study(direction  = CFG.metric_obj, 
                                    study_name = "OptunaEnsemble", 
                                    sampler    = TPESampler(seed = CFG.state)
                                   );
        study.optimize(Objective, 
                       n_trials          = CFG.ntrials, 
                       gc_after_trial    = True,
                       show_progress_bar = True
                      );
        
        weights          = study.best_params;  
        ens_weights[tgt] = weights;
        ens_score[tgt]   = np.round(study.best_value, 6);

        # Making weighted predictions on the test set:-
        sub_fl[tgt] = np.average(Mdl_Preds[cols], 
                                 weights = list(weights.values()),
                                 axis=1);
        del weights, ytrain, cols; 
        clear_output();
        
    PrintColor(f"\n--> Post ensemble weights");
    pprint(ens_weights, indent = 5, width = 10, depth = 2);
    PrintColor(f"\n--> Post ensemble score");
    pprint(ens_score, indent = 5, width = 10, depth = 2);
    
    sub_fl.to_csv(f"Submission_V{CFG.version_nb}.csv", index = None);
         
collect();
print();

In [ ]:
%%time 

if CFG.ML == "Y":  
    OOF_Preds.add_prefix(f"V{CFG.version_nb}_").to_csv(f"OOF_Preds_V{CFG.version_nb}.csv");
    Mdl_Preds.add_prefix(f"V{CFG.version_nb}_").to_csv(f"Mdl_Preds_V{CFG.version_nb}.csv"); 
    if isinstance(Scores, pd.DataFrame) == True:
        Scores.to_csv(f"Scores_V{CFG.version_nb}.csv");
           
collect();
print();

<a id="9"></a>
# <div style= "font-family: Cambria; font-weight:bold; letter-spacing: 0px; color:#ffffff; font-size:120%; text-align:left;padding:3.0px; background: #0052cc; border-bottom: 8px solid #cc9966" > NEXT STEPS<br> <div> 

<div style= "font-family: Cambria; letter-spacing: 0px; color:#000000; font-size:110%; text-align:left;padding:3.0px; background: #f2f2f2" >
1. Better feature engineering- this includes feature importance assessments, decision to include/ exclude features and new feature creation<br>
2. Better experiments with scaling, encoding with categorical columns. This seems to have some promise<br>
3. Better model tuning<br>
4. Model calibration- this may not be absolutely necessary with a rank-metric like GINI<br>
5. Adding more algorithms and new methods to the model suite<br>
6. Better ensemble strategy<br>
7. Do we exclude EC3- EC6??<br>
8. Do we treat this as a multi-label challenge only??
</div>